In [ ]:
import nest_asyncio
nest_asyncio.apply()

from wakis import WakeSolver
from wakis import SolverFIT3D
from wakis import GridFIT3D 
#from  clara_gridFIT3D_markCellsinSTL_WIP import GridFIT3D
#from wakis import clara_gridFIT3D_markCellsinSTL_WIP 
#from wakis.clara_gridFIT3D import GridFIT3D

import numpy as np
import pyvista as pv

from pyvista.trame.jupyter import launch_server
await launch_server().ready

#pv.set_jupyter_backend('html') 
pv.set_jupyter_backend('trame') 

from benchmark import benchmark

import pickle

In [ ]:
stl_finger= 'clara_stl_files/012_LHC_WVM_6L2-cube.stl'
surf_finger= pv.read(stl_finger)


stl_materials = {'Fingers': [1e4, 1., 1e4]}
stl_names = {'Fingers': stl_finger}
basename = 'Fingers'
stl_solids = {m: f'{stl_names[m]}' for m in stl_materials}


xmin, xmax, ymin, ymax, zmin, zmax = surf_finger.bounds
Lx, Ly, Lz = (xmax-xmin), (ymax-ymin), (zmax-zmin)
stl_tol=1e-3



Running the benchmark

In [ ]:

from concurrent.futures import ProcessPoolExecutor


'''# the worker function
def run_benchmark_task(params):
    m, r, xmin, xmax, ymin, ymax, zmin, zmax, stl_solids, stl_materials, surf_shell = params
    
    Nx, Ny, Nz = map(int, r.split('x'))
    spacing = [(xmax - xmin) / Nx, (ymax - ymin) / Ny, (zmax - zmin) / Nz]
    x, y, z = np.linspace(xmin, xmax, Nx), np.linspace(ymin, ymax, Ny), np.linspace(zmin, zmax, Nz)
    
    gfit = GridFIT3D(xmin, xmax, ymin, ymax, zmin, zmax, Nx, Ny, Nz, 
                     stl_solids=stl_solids, stl_materials=stl_materials, stl_scale=1.0,
                     stl_method=m)
    
    test_grid = gfit.grid
    
    spacing = [gfit.x[1] - gfit.x[0], gfit.y[1] - gfit.y[0], gfit.z[1] - gfit.z[0]]
    key=test_grid.array_names[0]

    vol, area, errvol, errarea = benchmark(test_grid, spacing, surf_shell,key)
    return m, r, spacing, {'vol': vol, 'vol_err': errvol, 'area': area, 'area_err': errarea}

if __name__ == '__main__':
    #methods = [ 'interior_points', 'implicit_distance_tol', 'voxelize_rectilinear']
    methods = ['voxelize_rectilinear']
    #resolutions = ['25x25x50', '50x50x100','75x75x150', '100x100x200','112x112x225', '125x125x250','135x135x270', '150x150x300', '160x160x320','175x175x350','185x185x370', '200x200x400','212x212x425','250x250x500', '260x260x520', '280x280x560','300x300x600','325x325x650','350x350x700']
    resolutions= ['25x25x25', '50x50x50','75x75x75',
                 '100x100x100','125x125x125', '150x150x150', '175x175x175', '200x200x200',
                   '250x250x250', '275x275x275','300x300x300','325x325x325','350x350x350',
                   '375x375x375','400x400x400','425x425x425','450x450x450'
                   ]

    tasks = []
    for r in resolutions:
        for m in methods:
            tasks.append((m, r, xmin, xmax, ymin, ymax, zmin, zmax, 
                          stl_solids, stl_materials, surf_finger))

    results = {m: {} for m in methods}
    

    print(f"Starting benchmark on 1 cores for {len(tasks)} tasks...")
    with ProcessPoolExecutor(max_workers=1) as executor:
        for m, r, spacing, data in executor.map(run_benchmark_task, tasks):
            results[m][r] = data
            print(f"Finished: {m} at {r}")

    
    filename = 'clara_gridres_benchmark_results_4.pkl'
    with open(filename, 'wb') as f:
        pickle.dump(results, f)

    print(f"Successfully saved results to {filename}")'''

Testing with smart mesh / mesh refinement the smallest detail of an STL object

In [ ]:
'''
spacing=np.min([x[1]-x[0], y[1]-y[0], z[1]-z[0]])
grid = pv.RectilinearGrid(x, y, z)
grid.compute_implicit_distance(surf_finger, inplace=True)
grid['Fingers']= grid.point_data_to_cell_data()['implicit_distance'] <= spacing 
inside_voxels = grid.threshold(0.5, scalars="Fingers")  
key='Fingers
benchmark(testgrid, spacing, surf_finger, key)'''

In [1]:
import nest_asyncio
nest_asyncio.apply()

from wakis import WakeSolver
from wakis import SolverFIT3D
from wakis import GridFIT3D 
#from  clara_gridFIT3D_markCellsinSTL_WIP import GridFIT3D
#from wakis import clara_gridFIT3D_markCellsinSTL_WIP 
#from wakis.clara_gridFIT3D import GridFIT3D

import numpy as np
import pyvista as pv

from pyvista.trame.jupyter import launch_server
await launch_server().ready

#pv.set_jupyter_backend('html') 
pv.set_jupyter_backend('trame') 

from benchmark import benchmark

import pickle

In [2]:
stl_finger='/home/cwimmelm/cernbox/wakis/clara_stl_files/012_LHC_WVM_6L2-cube.stl'
surf_finger= pv.read(stl_finger)

stl_materials = {'Fingers': [1e4, 1., 1e4]}
stl_names = {'Fingers': stl_finger}
basename = 'Fingers'
stl_solids = {m: f'{stl_names[m]}' for m in stl_materials}
print(stl_solids)

xmin, xmax, ymin, ymax, zmin, zmax = surf_finger.bounds
Lx, Ly, Lz = (xmax-xmin), (ymax-ymin), (zmax-zmin)
stl_tol=1e-3

Nx, Ny, Nz = 100, 100, 200
x = np.linspace(xmin, xmax, Nx)
y = np.linspace(ymin, ymax, Ny)
z = np.linspace(zmin, zmax, Nz)

{'Fingers': '/home/cwimmelm/cernbox/wakis/clara_stl_files/012_LHC_WVM_6L2-cube.stl'}


In [3]:
def read_stl(key):
        # import stl
        surf = pv.read(stl_solids[key])

        '''   # rotate
        surf = surf.rotate_x(stl_rotate[key][0])
        surf = surf.rotate_y(stl_rotate[key][1])
        surf = surf.rotate_z(stl_rotate[key][2])

        # translate
        surf = surf.translate(stl_translate[key])

        # scale
        surf = surf.scale(stl_scale[key])
        '''
        return surf

In [4]:
for key in stl_solids.keys():
    print(key)
    print(stl_solids[key])
    surf = read_stl(key)


Fingers
/home/cwimmelm/cernbox/wakis/clara_stl_files/012_LHC_WVM_6L2-cube.stl


In [5]:
dx,dy,dz=[5,5],[5,5],[5,5]

In [6]:
voxelize_spacing=np.array([np.min(dx),np.min(dy),np.min(dz)]) 
print(voxelize_spacing)

[5 5 5]


In [7]:
np.array([np.min(0.2),np.min(0.2),np.min(0.2)])

array([0.2, 0.2, 0.2])

In [8]:
voxelize_spacing=np.array([np.min(dx),np.min(dy),np.min(dz)]) 

grid = pv.StructuredGrid(x,y,z)
surf = surf_finger
#vox = surf.voxelize_rectilinear(spacing=voxelize_spacing)
vox = surf.voxelize_rectilinear(reference_volume=grid)

ValueError: Input point array shapes must match exactly

In [9]:
surf.voxelize_rectilinear(spacing=[0.2,0.2,0.2])

RectilinearGrid (0x7fdefc074100)
  N Cells:      166705840
  N Points:     167623749
  X Bounds:     -6.022e+01, 6.022e+01
  Y Bounds:     -4.387e+01, 7.657e+01
  Z Bounds:     1.790e+02, 2.710e+02
  Dimensions:   603, 603, 461
  N Arrays:     1

In [14]:
gfit_imp = GridFIT3D(xmin, xmax, ymin, ymax, zmin, zmax, Nx, Ny, Nz, stl_solids=stl_solids, stl_materials=stl_materials, stl_scale=1.0, stl_tol=stl_tol,
                     stl_method='voxelize_rectilinear')

Generating grid with 2000000 mesh cells...
Importing STL solids...
Default voxelization spacing used=np.mean([np.min(dx),np.min(dy),np.min(dz)])=0.9563412602742432
Total grid initialization time: 0.8260049819946289 s


In [ ]:


gfit_imp = GridFIT3D(xmin, xmax, ymin, ymax, zmin, zmax, Nx, Ny, Nz, stl_solids=stl_solids, stl_materials=stl_materials, stl_scale=1.0, stl_tol=stl_tol,
                     stl_method='voxelize_rectilinear',voxelize_spacing=np.array([0.2,0.2,0.2])
                    





Generating grid with 2000000 mesh cells...
Importing STL solids...
[!] Warning: voxelization for stl solid Fingers failed. Consider checking if the grid is uniform, subdividing the STL file or using a different method.


KeyError: 'Data array (Fingers) not present in this dataset.'

In [ ]:
# Assuming your object is named 'grid'
gfit_imp.plot_stl_mask(stl_solid='Fingers')

In [ ]:


gfit_imp = GridFIT3D(xmin, xmax, ymin, ymax, zmin, zmax, Nx, Ny, Nz, stl_solids=stl_solids, stl_materials=stl_materials, stl_scale=1.0, stl_tol=stl_tol,
                     #stl_method='voxelize_rectilinear',
                     use_mesh_refinement=True)





In [ ]:
gfit_imp.dz

analysing the distribution of distances between the snap points = feature spacing

In [ ]:
edges = surf_finger.extract_feature_edges(boundary_edges=True, manifold_edges=False)
snap_tol=1e-8
# Extract points lying in the X-Z plane (Y ≈ 0)
xz_plane_points = edges.points[np.abs(edges.points[:, 1]) < snap_tol]
# Extract points lying in the Y-Z plane (X ≈ 0)
yz_plane_points = edges.points[np.abs(edges.points[:, 0]) < snap_tol]
# Extract points lying in the X-Y plane (Z ≈ 0)
xy_plane_points = edges.points[np.abs(edges.points[:, 2]) < snap_tol]

snap_points = np.r_[xz_plane_points, yz_plane_points, xy_plane_points]

# get the unique x, y, z coordinates
x_snaps = np.unique(np.round(snap_points[:, 0], 5))
y_snaps = np.unique(np.round(snap_points[:, 1], 5))
z_snaps = np.unique(np.round(snap_points[:, 2], 5))


In [ ]:
x_snaps

In [ ]:
print(f"Total points in edges: {edges.n_points}")

In [ ]:
import matplotlib.pyplot as plt

# Assuming you have already run _compute_snap_points
spacings = {
    'X': np.diff(x_snaps),
    'Y': np.diff(y_snaps),
    'Z': np.diff(z_snaps)
}

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)

for i, (axis, vals) in enumerate(spacings.items()):
    axes[i].hist(vals, bins=50, color='salmon', edgecolor='black')
    axes[i].set_title(f'Snap Spacing Distribution ({axis})')
    axes[i].set_xlabel('Distance')
    axes[i].set_yscale('log') # Useful if you have many large and few small gaps

plt.tight_layout()
plt.show()

In [ ]:
stl_finger='/home/cwimmelm/cernbox/wakis/clara_stl_files/012_LHC_WVM_6L2-cube.stl'
surf_finger= pv.read(stl_finger)

xmin, xmax, ymin, ymax, zmin, zmax = surf_finger.bounds
Lx, Ly, Lz = (xmax-xmin), (ymax-ymin), (zmax-zmin)
stl_tol=1e-3
Nx, Ny, Nz = 100, 100, 200
x = np.linspace(xmin, xmax, Nx)
y = np.linspace(ymin, ymax, Ny)
z = np.linspace(zmin, zmax, Nz)
spacing= [x[1] - x[0], y[1] - y[0], z[1] - z[0]]

In [ ]:
edges = surf_finger.extract_feature_edges(boundary_edges=True, manifold_edges=False)
snap_tol=1e-8
points = edges.points
lines = edges.lines.reshape(-1, 3) #assuming 2 points +1 padding

p1 = points[lines[:, 1]]
p2 = points[lines[:, 2]]
edge_lengths = np.linalg.norm(p1 - p2, axis=1)

real_edges = edge_lengths[edge_lengths > snap_tol] #filter out numerical noise
smallest_geometric_detail = np.min(real_edges)
small_edges = real_edges[real_edges <= 0.7]
mean_geometric_detail = np.mean(small_edges)

In [ ]:
print(mean_geometric_detail)
print(f"The smallest geometric detail detected is: {smallest_geometric_detail}")
print(np.min(real_edges)*np.max(real_edges)*0.5)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. Get the points and the line connectivity from your 'edges' object
points = edges.points
# lines is a padded array: [2, p1_idx, p2_idx, 2, p3_idx, p4_idx, ...]
lines = edges.lines.reshape(-1, 3) 

# 2. Calculate Euclidean distance for every edge
p1 = points[lines[:, 1]]
p2 = points[lines[:, 2]]
dist = np.linalg.norm(p1 - p2, axis=1)

# 3. Filter out zero-length edges (numerical noise)
dist = dist[dist > 1e-9]

# 4. Plot the Distribution
plt.figure(figsize=(10, 6))
plt.hist(dist, bins=100, color='skyblue', edgecolor='black', log=True) # log=True helps see small details
plt.axvline(np.min(dist), color='red', linestyle='--', label=f'Min Detail: {np.min(dist):.4f}')
plt.axvline(np.mean(dist), color='green', linestyle='--', label=f'Mean Detail: {np.mean(dist):.4f}')

plt.title('Distribution of STL Geometric Feature Lengths')
plt.xlabel('Edge Length (Geometric Detail Size)')
plt.ylabel('Frequency (Log Scale)')
plt.legend()
plt.grid(True, which="both", ls="-", alpha=0.2)
plt.show()

In [ ]:
# Calculate the distance between adjacent snap points
dx_features = np.diff(x_snaps)
dy_features = np.diff(y_snaps)
dz_features = np.diff(z_snaps)

# Find the "Critical Detail" size
smallest_feature = np.min([np.min(dx_features), np.min(dy_features), np.min(dz_features)])

print(f"The smallest geometric detail detected is: {smallest_feature}")

In [ ]:
edges